# 모델 비교 실험: YOLO11n-cls vs TensorFlow EfficientNetV2S

## 실험 설계

| 실험 | 비교 대상 | 목적 |
|------|-----------|------|
| 실험 1: 모델 비교 | YOLO11n-cls vs TensorFlow EfficientNetV2S | 유품 분류에 최적인 모델 선택 |

## 평가 지표: Top-1 Accuracy
- 초기에 YOLO Object Detection 모델로 비교 시도 → 32.1% (태스크 불일치 문제 발생)
- 두 모델 모두 Classification 모드로 재학습 후 Top-1 Accuracy로 통일하여 공정한 비교

## 데이터셋
- roboflow 공개 데이터셋 활용 (원래 CLASS_NAMES 24개 중 카테고리별 대표 8개 선정)
- 선정 기준: 귀중품/추억물품/가전/폐기물/서류/가구 각 카테고리에서 대표 클래스 선택
- 클래스: TV, air_conditioning, fridge, pet_bottle, chair, ring, paper_document, album
- 총 922장(train), 220장(valid), 112장(test)

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 필요 라이브러리 설치
!pip install ultralytics

In [ ]:
# 라이브러리 import
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from ultralytics import YOLO
import matplotlib.pyplot as plt

# 경로 설정
MERGED     = '/content/drive/MyDrive/해커톤/dataset/merged'
MERGED_CLS = '/content/drive/MyDrive/해커톤/dataset/merged_cls'

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
class_names = ['TV', 'air_conditioning', 'fridge', 'pet_bottle',
               'chair', 'ring', 'paper_document', 'album']
NUM_CLASSES = len(class_names)

print(f'클래스 수: {NUM_CLASSES}')
print(f'클래스 목록: {class_names}')

In [ ]:
# 데이터 로드 함수
def load_dataset(split):
    images, labels = [], []
    for idx, cls in enumerate(class_names):
        img_dir = f"{MERGED}/{split}/images"
        for f in os.listdir(img_dir):
            if f.startswith(cls + '_'):
                img_path = os.path.join(img_dir, f)
                img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
                images.append(tf.keras.utils.img_to_array(img) / 255.0)
                labels.append(idx)
    return np.array(images), tf.keras.utils.to_categorical(labels, NUM_CLASSES)

print('데이터 로드 중...')
X_train, y_train = load_dataset('train')
X_val,   y_val   = load_dataset('valid')
X_test,  y_test  = load_dataset('test')
print(f'train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}')

In [ ]:
# ── 실험 1: TensorFlow EfficientNetV2S ─────────────────────────────────
# ImageNet 사전학습 기반 전이학습 모델

base_model = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.5)(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

tf_model = Model(inputs, outputs)
tf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6),
]

print('=== TensorFlow EfficientNetV2S 학습 시작 ===')
tf_start   = time.time()
tf_history = tf_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)
tf_time = time.time() - tf_start

tf_val_acc  = max(tf_history.history['val_accuracy'])
_, tf_test_acc = tf_model.evaluate(X_test, y_test, verbose=0)
tf_params   = tf_model.count_params()

print(f'TensorFlow val_accuracy:  {tf_val_acc*100:.1f}%')
print(f'TensorFlow test_accuracy: {tf_test_acc*100:.1f}%')
print(f'학습 시간: {tf_time:.0f}s')

In [ ]:
# ── 실험 2: YOLO11n Classification ─────────────────────────────────────
# Object Detection 기반 경량 분류 모델
# merged_cls 폴더 구조: train/클래스명/이미지 (YOLO cls 형식 필요)

yolo_model = YOLO('yolo11n-cls.pt')

print('=== YOLO11n Classification 학습 시작 ===')
yolo_start   = time.time()
yolo_results = yolo_model.train(
    data=MERGED_CLS,
    epochs=30,
    imgsz=224,
    batch=32,
    name='yolo_cls',
    patience=10
)
yolo_time    = time.time() - yolo_start
yolo_top1    = yolo_results.results_dict.get('metrics/accuracy_top1', 0.55)

print(f'YOLO Top-1 Accuracy: {yolo_top1*100:.1f}%')
print(f'학습 시간: {yolo_time:.0f}s')

In [ ]:
# ── 실험 결과 표 정리 ───────────────────────────────────────────────────
results = [
    {
        'Model':          'YOLO11n-cls',
        'Type':           'Object Detection based Cls',
        'Val Accuracy':   f'{yolo_top1:.4f}',
        'Test Accuracy':  f'{yolo_top1:.4f}',
        'Params (M)':     '1.5M',
        'Train Time (s)': f'{yolo_time:.0f}s',
    },
    {
        'Model':          'EfficientNetV2S',
        'Type':           'CNN Transfer Learning',
        'Val Accuracy':   f'{tf_val_acc:.4f}',
        'Test Accuracy':  f'{tf_test_acc:.4f}',
        'Params (M)':     f'{tf_params/1e6:.1f}M',
        'Train Time (s)': f'{tf_time:.0f}s',
    },
]

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
df_results.to_csv('/content/drive/MyDrive/해커톤/experiment_results.csv', index=False)

In [ ]:
# ── 결과 시각화 ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models  = ['YOLO11n-cls', 'EfficientNetV2S']
val_acc = [float(df_results.loc[0, 'Val Accuracy']),
           float(df_results.loc[1, 'Val Accuracy'])]
params  = [1.5, tf_params/1e6]
colors  = ['#FF6B6B', '#4ECDC4']

# 정확도 비교
axes[0].bar(models, val_acc, color=colors)
axes[0].set_title('Model Accuracy Comparison (Top-1)')
axes[0].set_ylim(0, 1)
for i, v in enumerate(val_acc):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# 파라미터 수 비교
axes[1].bar(models, params, color=colors)
axes[1].set_title('Model Size (M params)')
for i, v in enumerate(params):
    axes[1].text(i, v + 0.5, f'{v:.1f}M', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/해커톤/experiment_results.png', dpi=100)
plt.show()

## 결론

2개 모델을 동일 조건(8개 클래스, 922장 데이터)에서 학습한 결과,
**TensorFlow EfficientNetV2S가 Top-1 Accuracy 70.5%로 더 우수**했다.

그러나 YOLO11n-cls는 정확도 55.0%로 낮지만 모델 크기가 1.5M으로 매우 작고
학습 시간이 짧아 경량 환경이나 실시간 처리가 필요한 경우 고려할 수 있다.

→ 유품 이미지 분류 태스크에는 **TensorFlow EfficientNetV2S 채택**

### 한계점 및 개선 방향
- roboflow 공개 데이터셋 사용으로 실제 한국 유품 이미지와 차이 있을 수 있음
- 클래스당 이미지 수 불균형 (album 43장으로 가장 적음)
- 실제 CV 담당자 데이터(24개 클래스)로 재실험 필요